# M06-P02 — LangChain RAG & Evaluation

En M06-P01 construimos RAG sin frameworks para ver todas las piezas. Ahora reconstruimos el mismo patrón con LangChain y añadimos algo esencial: **evaluación de retrieval**.

```text
Document → text splitter → Chroma → retrieval → evaluación → prompt → Bedrock
```

La pregunta principal es: **¿cómo sabemos si nuestro retriever funciona bien?**

In [ ]:
import json
from pathlib import Path
import pandas as pd

from enterprise_genai.langchain_rag import (
    CourseBedrockEmbeddings,
    answer_with_langchain,
    build_chroma,
    evaluate_retrieval,
    retrieve_with_scores,
    split_documents,
    to_langchain_documents,
)

CORPUS_PATH = Path('data/knowledge/m05_enterprise_chunks.json')
EVAL_PATH = Path('data/evaluation/m06_rag_eval.json')
records = json.loads(CORPUS_PATH.read_text(encoding='utf-8'))
eval_cases = json.loads(EVAL_PATH.read_text(encoding='utf-8'))

print('Documentos fuente:', len(records))
print('Casos de evaluación:', len(eval_cases))

## 1. De nuestros diccionarios a `Document`

LangChain usa `Document` para combinar `page_content` y `metadata`. El significado no cambia respecto a P01: adoptamos una abstracción estándar.

In [ ]:
lc_documents = to_langchain_documents(records)
print('LangChain Documents:', len(lc_documents))
print('Tipo:', type(lc_documents[0]).__name__)
print('Contenido del primero:')
print(lc_documents[0].page_content)
print('\nMetadata:')
print(lc_documents[0].metadata)

## 2. Dos estrategias de chunking

En P01 los documentos eran tan cortos que 80 documentos producían 80 chunks. Aquí forzamos una comparación real con `RecursiveCharacterTextSplitter`.

```text
chunks grandes  → 300 caracteres, overlap 40
chunks pequeños → 120 caracteres, overlap 30
```

No buscamos una configuración universalmente mejor: queremos observar que chunking es una decisión que puede medirse.

In [ ]:
large_chunks = split_documents(lc_documents, chunk_size=300, chunk_overlap=40)
small_chunks = split_documents(lc_documents, chunk_size=120, chunk_overlap=30)

print('Documentos fuente:', len(lc_documents))
print('Chunks grandes:', len(large_chunks))
print('Chunks pequeños:', len(small_chunks))

display(pd.DataFrame([
    {'estrategia': 'large_300_40', 'chunk_size': 300, 'overlap': 40, 'chunks': len(large_chunks)},
    {'estrategia': 'small_120_30', 'chunk_size': 120, 'overlap': 30, 'chunks': len(small_chunks)},
]))

In [ ]:
example_id = None
for doc in lc_documents:
    count = sum(1 for chunk in small_chunks if chunk.metadata['document_id'] == doc.metadata['document_id'])
    if count > 1:
        example_id = doc.metadata['document_id']
        break

print('Documento de ejemplo:', example_id)
for i, chunk in enumerate([c for c in small_chunks if c.metadata['document_id'] == example_id], start=1):
    print(f'\n--- chunk {i} ---')
    print(chunk.page_content)
    print(chunk.metadata)

## 3. Índices Chroma

LangChain conecta los chunks con un vector store. Usamos Chroma en memoria y el mismo Titan de las prácticas anteriores. `CourseBedrockEmbeddings` adapta nuestros helpers al contrato `Embeddings` de LangChain y conserva la concurrencia para no hacer la ingesta innecesariamente lenta.

In [ ]:
embeddings = CourseBedrockEmbeddings(max_workers=8)

large_store = build_chroma(large_chunks, collection_name='m06_p02_large', embeddings=embeddings)
small_store = build_chroma(small_chunks, collection_name='m06_p02_small', embeddings=embeddings)

print('Índice large listo:', len(large_chunks), 'chunks')
print('Índice small listo:', len(small_chunks), 'chunks')

## 4. Una consulta antes de evaluar

Comprobamos una búsqueda conocida. Ahora Chroma/LangChain realizan retrieval en lugar de nuestro bucle manual de similitud.

In [ ]:
QUERY = '¿Qué debo comprobar si no recibo el segundo factor al conectarme por VPN?'
results = retrieve_with_scores(small_store, QUERY, k=4)

rows = []
for item in results:
    doc = item['document']
    rows.append({
        'rank': item['rank'],
        'score': round(item['score'], 4),
        'document_id': doc.metadata['document_id'],
        'service': doc.metadata['service'],
        'title': doc.metadata['title'],
        'chunk': doc.page_content,
    })
display(pd.DataFrame(rows))

## 5. Benchmark etiquetado

Una demo con una sola pregunta puede engañarnos. Usamos 10 consultas para las que conocemos el documento que debería recuperarse.

In [ ]:
display(pd.DataFrame(eval_cases))

## 6. Hit Rate@k y MRR@k

**Hit Rate@k** pregunta si el documento correcto aparece entre los `k` primeros. **MRR@k** premia que aparezca lo más arriba posible.

```text
rank 1 → 1.0
rank 2 → 0.5
rank 3 → 0.333...
no aparece → 0
```

In [ ]:
experiments = []
for strategy_name, store in [('large_300_40', large_store), ('small_120_30', small_store)]:
    for k in [1, 3, 5]:
        result = evaluate_retrieval(store, eval_cases, k=k)
        experiments.append({
            'estrategia': strategy_name,
            'top_k': k,
            'filtro_service': False,
            'hit_rate': round(result['hit_rate'], 3),
            'mrr': round(result['mrr'], 3),
            'hits': result['hits'],
            'casos': result['cases'],
        })
evaluation_df = pd.DataFrame(experiments)
display(evaluation_df)

## 7. Metadata filtering

Si la aplicación ya conoce el servicio, podemos combinar similitud semántica con una restricción estructurada `service`. Comparamos retrieval global y filtrado.

In [ ]:
filtered = evaluate_retrieval(small_store, eval_cases, k=3, use_service_filter=True)
unfiltered = evaluate_retrieval(small_store, eval_cases, k=3, use_service_filter=False)

display(pd.DataFrame([
    {'configuración': 'sin filtro', 'top_k': 3, 'hit_rate': round(unfiltered['hit_rate'], 3), 'mrr': round(unfiltered['mrr'], 3)},
    {'configuración': 'service filter', 'top_k': 3, 'hit_rate': round(filtered['hit_rate'], 3), 'mrr': round(filtered['mrr'], 3)},
]))

## 8. Evaluación caso por caso

Las métricas agregadas no explican por qué falla una consulta. Abrimos la evaluación para ver documento esperado, primer resultado y rank.

In [ ]:
detail_df = pd.DataFrame(filtered['rows'])
display(detail_df[['id', 'expected_document_id', 'top_document_id', 'hit', 'rank', 'reciprocal_rank']])

## 9. RAG con LCEL

Una vez evaluado retrieval, construimos generación con LangChain Expression Language:

```text
ChatPromptTemplate | ChatBedrockConverse | StrOutputParser
```

In [ ]:
rag_results = retrieve_with_scores(small_store, QUERY, k=4, metadata_filter={'service': 'VPN'})
generated = answer_with_langchain(QUERY, rag_results)
print(generated['answer'])

## 10. Contexto que recibió la cadena

LangChain reduce boilerplate, pero el patrón sigue siendo `retrieval → contexto → prompt → modelo`. Abrimos de nuevo la caja.

In [ ]:
print(generated['context'])

## 11. Qué ha cambiado respecto a M06-P01

LangChain aporta `Document`, `TextSplitter`, `VectorStore`, `PromptTemplate`, `ChatModel`, LCEL y parsers. Pero sigue siendo responsabilidad de la aplicación decidir corpus, chunking, metadata, top-k, filtros, evaluación y grounding.

> Un RAG no debería considerarse correcto porque “parece responder bien”. El retrieval se puede y se debe medir.